# 02 — Cleaning, Feature Engineering & Leakage-Safe Preprocessing

**Goriška Brda Harvest Prediction App**

Builds directly on the leakage findings from `01_eda.ipynb`. This notebook:
1. applies the approved leakage-column exclusions,
2. engineers new features from the safe columns,
3. splits the data (train/val/test) before anything is fit,
4. runs a leakage-safe correlation filter,
5. builds the `ColumnTransformer` preprocessing pipeline and fits it **only on the
   training split**.

No model is trained here — that's Phase 3. All reusable logic lives in
`analysis/src/preprocessing.py`; this notebook calls it and inspects the results.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))
from data import load_dataset
from preprocessing import (
    ALL_FEATURES,
    BASE_NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    CLASSIFICATION_TARGET,
    CORRELATED_FEATURES_DROPPED,
    ENGINEERED_NUMERIC_FEATURES,
    LEAKAGE_COLUMNS,
    NUMERIC_FEATURES,
    REGRESSION_TARGET,
    build_preprocessor,
    engineer_features,
    find_correlated_features,
    train_val_test_split,
)

pd.set_option("display.max_columns", None)

df = load_dataset()
df.shape


(2176, 38)

## 1. Apply the approved leakage exclusions


In [2]:
print("Dropping:", LEAKAGE_COLUMNS)
df = df.drop(columns=LEAKAGE_COLUMNS)
print("shape after dropping leakage columns:", df.shape)
assert CLASSIFICATION_TARGET in df.columns and REGRESSION_TARGET in df.columns


Dropping: ['harvest_doy', 'harvest_date', 'yield_category', 'gbai_score', 'ln_quality_index', 'est_price_eur', 'vintage_quality', 'climate_risk', 'harvest_rainfall_mm', 'autumn_rainfall_mm', 'annual_rainfall_mm', 'grape_color', 'vintage_age_years']
shape after dropping leakage columns: (2176, 25)


Both targets are still present — we only removed *inputs* that leaked the outcome, never
the outcomes themselves. `df.shape` going from 38 to 25 columns matches 13 dropped columns
(the 11 leakage columns already covered in EDA sections 3-6, plus `grape_color` and
`vintage_age_years` from section 7).


## 2. Engineer new features


In [3]:
df = engineer_features(df)
print("shape after engineering:", df.shape)
df[ENGINEERED_NUMERIC_FEATURES].describe()


shape after engineering: (2176, 31)


,heat_stress_index,rainfall_efficiency,frost_heat_ratio,gdd_per_sunshine_hour,yield_lag_change,harvest_lag_change
count,2176.000000,2176.000000,2176.000000,2176.000000,2004.000000,2112.000000
mean,28202.529412,0.405661,0.242123,0.664588,-18.902196,-0.372159
std,10726.689995,0.067964,0.273820,0.071820,1149.128908,6.893845
min,10976.000000,0.236628,0.000000,0.481695,-3770.000000,-22.000000
25%,21675.000000,0.356554,0.000000,0.631967,-750.000000,-5.000000
50%,28267.500000,0.411599,0.149074,0.678662,60.000000,0.000000
75%,32277.000000,0.461370,0.375000,0.701661,700.000000,4.000000
max,60830.000000,0.527337,1.222222,0.800000,4310.000000,23.000000


Six engineered features, each built only from columns already cleared as safe in Phase 1:

| Feature | Formula | Rationale |
|---|---|---|
| `heat_stress_index` | `summer_heat_days * growing_degree_days` | compounding heat exposure |
| `rainfall_efficiency` | `(spring+summer rainfall) / growing_degree_days` | water received per unit of heat accumulated (drought-stress proxy) |
| `frost_heat_ratio` | `spring_frost_days / summer_heat_days` | frost risk share relative to heat exposure |
| `gdd_per_sunshine_hour` | `growing_degree_days / sunshine_hours` | heat accumulation efficiency per hour of daylight |
| `yield_lag_change` | `prev_yield_kg_ha - (yield two years ago)` | short-term yield momentum |
| `harvest_lag_change` | `prev_harvest_doy - (harvest day two years ago)` | short-term harvest-timing drift |

`yield_lag_change`/`harvest_lag_change` are built **only** from `prev_yield_kg_ha` /
`prev_harvest_doy` (already-validated lag-1 columns), shifted one row further back within
each (location, grape_variety) history. That never touches the current row's own
`yield_kg_ha`/`harvest_doy` — see `engineer_features()` docstring for the full reasoning.


In [4]:
new_nans = df[NUMERIC_FEATURES].isna().sum()
new_nans[new_nans > 0]


min_spring_temp_C      62
humidity_pct           73
soil_moisture_pct      57
prev_yield_kg_ha       56
yield_lag_change      172
harvest_lag_change     64
dtype: int64

`yield_lag_change` (172 missing) and `harvest_lag_change` (64 missing) have more gaps than
the raw lag columns they're built from — expected, since they need *two* prior years of
history, and the first two years of each (location, grape_variety) series can't supply
that. This is structural, not a data quality problem, and will be handled by the same
median-imputation strategy as everything else — **inside the pipeline, fit on the training
split only** (section 5).


## 3. Split before fitting anything

The split happens **before** the correlation filter and **before** the preprocessing
pipeline — both of the next two steps must only ever see the training rows.


In [5]:
train_df, val_df, test_df = train_val_test_split(df)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")

for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    props = split[CLASSIFICATION_TARGET].value_counts(normalize=True).round(3)
    print(name, dict(props))


train=1522  val=327  test=327
train {'Early': np.float64(0.349), 'Normal': np.float64(0.334), 'Late': np.float64(0.317)}
val {'Early': np.float64(0.349), 'Normal': np.float64(0.336), 'Late': np.float64(0.315)}
test {'Early': np.float64(0.349), 'Normal': np.float64(0.333), 'Late': np.float64(0.318)}


In [6]:
# Sanity check: we're reusing one split (stratified on harvest_category) for the
# regression target too. Confirm yield_kg_ha isn't accidentally skewed across splits.
for name, split in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: mean={split[REGRESSION_TARGET].mean():.0f}  std={split[REGRESSION_TARGET].std():.0f}")


train: mean=5314  std=1110
val: mean=5325  std=989
test: mean=5362  std=1072


Class balance (Early/Normal/Late) is nearly identical across all three splits, as expected
from stratification. `yield_kg_ha` mean/std are also close across splits (within ~50
kg/ha of each other) even though we didn't stratify on it directly — confirms a **second,
separate split isn't needed** for the regression task. Using one split for both targets
keeps the two models directly comparable (same vineyards/years in each partition) and
avoids maintaining two parallel datasets.


In [7]:
X_train = train_df[ALL_FEATURES]
X_val = val_df[ALL_FEATURES]
X_test = test_df[ALL_FEATURES]

y_train_clf, y_val_clf, y_test_clf = train_df[CLASSIFICATION_TARGET], val_df[CLASSIFICATION_TARGET], test_df[CLASSIFICATION_TARGET]
y_train_reg, y_val_reg, y_test_reg = train_df[REGRESSION_TARGET], val_df[REGRESSION_TARGET], test_df[REGRESSION_TARGET]

X_train.shape, X_val.shape, X_test.shape


((1522, 24), (327, 24), (327, 24))

## 4. Leakage-safe correlation filter (feature selection)

Run against the **pre-filter** candidate set (`train_df`, not the already-filtered
`X_train` built above) — this reproduces the decision baked into
`preprocessing.NUMERIC_FEATURES` from the raw training data, rather than assuming it.


In [8]:
candidate_numeric = BASE_NUMERIC_FEATURES + ENGINEERED_NUMERIC_FEATURES
dropped = find_correlated_features(train_df[candidate_numeric], candidate_numeric, threshold=0.9)
dropped


['growing_season_temp_C',
 'temp_deviation_C',
 'growing_degree_days',
 'heat_stress_index',
 'frost_heat_ratio']

In [9]:
assert set(dropped) == set(CORRELATED_FEATURES_DROPPED), "recompute preprocessing.py's hardcoded list if this changes"

corr = train_df[candidate_numeric].corr()
for col in dropped:
    partner = corr[col].drop(col).abs().idxmax()
    print(f"{col:24s} <-> {partner:24s}  r={corr.loc[col, partner]:.3f}")


growing_season_temp_C    <-> avg_temperature_C         r=1.000
temp_deviation_C         <-> avg_temperature_C         r=1.000
growing_degree_days      <-> temp_deviation_C          r=1.000
heat_stress_index        <-> summer_heat_days          r=0.954
frost_heat_ratio         <-> spring_frost_days         r=0.919


**Why this is leakage-safe:** `find_correlated_features()` only ever inspects `X_train`
(feature-to-feature correlation) — it never touches `y_train`, and it's computed
exclusively on the training split. The resulting drop-list is then applied identically to
`X_val`/`X_test` by simply not selecting those columns; nothing about the validation or
test rows influenced the decision.

**What it found**, on real training data: `growing_season_temp_C`, `temp_deviation_C`, and
`growing_degree_days` are all **r=1.000** with `avg_temperature_C` — this dataset derives
them from the same single temperature signal. `heat_stress_index` (r=0.954) and
`frost_heat_ratio` (r=0.919) — two of the features engineered in section 2 — turn out to
be near-duplicates of `summer_heat_days` and `spring_frost_days` respectively, once the
temperature duplicates collapse. That's a legitimate finding, not a mistake: engineering a
feature and then dropping it after feature selection shows the pipeline is *not* just
keeping everything on faith.

This list is already baked into `preprocessing.NUMERIC_FEATURES` (with the reasoning
documented inline) — the assertion above confirms the notebook reproduces that decision
from the raw training data rather than the module asserting it blindly.


## 5. Final feature list


In [10]:
print(f"Categorical ({len(CATEGORICAL_FEATURES)}):", CATEGORICAL_FEATURES)
print(f"\nNumeric, post-filter ({len(NUMERIC_FEATURES)}):")
for f in NUMERIC_FEATURES:
    tag = "(engineered)" if f in ENGINEERED_NUMERIC_FEATURES else ""
    print(" -", f, tag)


Categorical (3): ['grape_variety', 'soil_type', 'location']

Numeric, post-filter (21):
 - year 
 - elevation_m 
 - vine_age_years 
 - avg_temperature_C 
 - min_spring_temp_C 
 - summer_heat_days 
 - spring_frost_days 
 - winter_rainfall_mm 
 - spring_rainfall_mm 
 - summer_rainfall_mm 
 - rainfall_deviation_mm 
 - humidity_pct 
 - sunshine_hours 
 - soil_moisture_pct 
 - heat_frost_ratio 
 - prev_harvest_doy 
 - prev_yield_kg_ha 
 - rainfall_efficiency (engineered)
 - gdd_per_sunshine_hour (engineered)
 - yield_lag_change (engineered)
 - harvest_lag_change (engineered)


## 6. Build the pipeline and fit on the training split only


In [11]:
preprocessor = build_preprocessor(NUMERIC_FEATURES, CATEGORICAL_FEATURES)

# Fit happens exactly once, on X_train. Val/test are only ever .transform()-ed.
preprocessor.fit(X_train)

X_train_transformed = preprocessor.transform(X_train)
X_val_transformed = preprocessor.transform(X_val)
X_test_transformed = preprocessor.transform(X_test)

X_train_transformed.shape, X_val_transformed.shape, X_test_transformed.shape


((1522, 43), (327, 43), (327, 43))

In [12]:
# Prove the imputer's learned statistics come only from the training rows.
numeric_imputer = preprocessor.named_transformers_["numeric"].named_steps["imputer"]
pd.Series(numeric_imputer.statistics_, index=NUMERIC_FEATURES).loc[["prev_yield_kg_ha", "humidity_pct", "soil_moisture_pct"]]


prev_yield_kg_ha     5380.0
humidity_pct           66.8
soil_moisture_pct      37.5
dtype: float64

In [13]:
categorical_encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]
categorical_encoder.get_feature_names_out(CATEGORICAL_FEATURES)


array(['grape_variety_Cabernet Sauvignon', 'grape_variety_Chardonnay',
       'grape_variety_Malvazija', 'grape_variety_Merlot',
       'grape_variety_Pinot Grigio', 'grape_variety_Pinot Noir',
       'grape_variety_Rebula', 'grape_variety_Sauvignon Blanc',
       'soil_type_Clay', 'soil_type_Clay-Loam', 'soil_type_Loam',
       'soil_type_Marl', 'soil_type_Marl-Loam', 'soil_type_Sandy-Loam',
       'location_Biljana', 'location_Cerovo', 'location_Dobrovo',
       'location_Kozana', 'location_Medana', 'location_Neblo',
       'location_Vipolže', 'location_Šmartno'], dtype=object)

The `ColumnTransformer` was fit exactly once, on `X_train`. `X_val`/`X_test` only ever go
through `.transform()`, never `.fit()` or `.fit_transform()` — so the imputation
medians/modes and the scaler's median/IQR are computed purely from training rows, and the
one-hot categories are learned from training rows too (`handle_unknown="ignore"` protects
against a category appearing in val/test that wasn't seen in training, which is a realistic
production scenario — a new grape variety or soil type could show up later).

After one-hot expansion, the final feature matrix has


In [14]:
print(f"{X_train_transformed.shape[1]} columns "
      f"({len(NUMERIC_FEATURES)} numeric + "
      f"{X_train_transformed.shape[1] - len(NUMERIC_FEATURES)} one-hot categorical)")


43 columns (21 numeric + 22 one-hot categorical)


## Summary

- Loaded the raw dataset (2,176 rows, 38 columns), dropped the 13 approved leakage/redundant
  columns → 25 columns.
- Engineered 6 new features from already-safe columns; 2 (`heat_stress_index`,
  `frost_heat_ratio`) were later dropped by the correlation filter as near-duplicates of
  simpler features — kept as a documented, honest part of the process rather than
  quietly removed.
- Split into train (1,522) / val (327) / test (327), stratified on `harvest_category`;
  confirmed the same split keeps `yield_kg_ha` balanced too, so one split serves both tasks.
- Ran a correlation filter (fit on the training split only) that additionally dropped 3
  raw columns (`growing_season_temp_C`, `temp_deviation_C`, `growing_degree_days`) found to
  be perfectly collinear (r=1.000) with `avg_temperature_C`.
- Final feature set: 3 categorical + 17 numeric, expanded to


In [15]:
X_train_transformed.shape[1]


43

 columns after one-hot encoding.
- Built the `ColumnTransformer` (median/RobustScaler for numeric, most-frequent/OneHotEncoder
  for categorical) and fit it **only on the training split**, verified above.

**Stopping here — no model training in this notebook**, per the phase plan. Ready for your
review before Phase 3 (model training and honest comparison).
